In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('IPL_cleaned_dataset1.csv')

In [3]:
df.head()

,Unnamed: 0,match_id,batting_team,bowling_team,over,runs_target,toss_decision,venue,city,overs,balls_left,runs_left,wickets_left,current_run_rate,required_run_rate,result,toss_advantage
0,1751,335989,Mumbai Indians,Chennai Super Kings,0,209.0,field,"MA Chidambaram Stadium, Chepauk",Chennai,20,119,209.0,10,0.0,10.54,0,1
1,1752,335989,Mumbai Indians,Chennai Super Kings,0,209.0,field,"MA Chidambaram Stadium, Chepauk",Chennai,20,119,208.0,10,6.0,10.49,0,1
2,1753,335989,Mumbai Indians,Chennai Super Kings,0,209.0,field,"MA Chidambaram Stadium, Chepauk",Chennai,20,118,207.0,10,6.0,10.53,0,1
3,1754,335989,Mumbai Indians,Chennai Super Kings,0,209.0,field,"MA Chidambaram Stadium, Chepauk",Chennai,20,117,203.0,10,12.0,10.41,0,1
4,1755,335989,Mumbai Indians,Chennai Super Kings,0,209.0,field,"MA Chidambaram Stadium, Chepauk",Chennai,20,117,202.0,10,14.0,10.36,0,1


In [4]:


df = df.sort_values(by=["match_id", "over", "balls_left"], ascending=[True, True, False])


df_over = df.groupby(["match_id", "over"]).agg({

    # Match situation (take last ball of over)
    "runs_left": "last",
    "balls_left": "last",
    "wickets_left": "last",
    "current_run_rate": "last",

    # Static features
    "runs_target": "first",
    "batting_team": "first",
    "bowling_team": "first",
    "toss_decision": "first",
    "toss_advantage": "first",

    "result": "first"

}).reset_index()


df_over["required_run_rate"] = np.where(
    df_over["balls_left"] > 0,
    df_over["runs_left"] / (df_over["balls_left"] / 6),
    0
)


df_over["pressure"] = df_over["required_run_rate"] - df_over["current_run_rate"]


df_over.replace([np.inf, -np.inf], np.nan, inplace=True)
df_over.dropna(inplace=True)

#

In [5]:
full_overs = df_over['balls_left'] // 6
    
rem_balls = df_over['balls_left'] % 6
    
    
df_over['over_left'] = full_overs + (rem_balls / 10)

In [6]:
df_over['is_tail_batting']=(df_over['wickets_left']<=4).astype(int)

In [7]:
df_over['wickets_fallen']=10-df_over['wickets_left']

In [8]:
df_over["phase"] = pd.cut(
    df_over["over"],
    bins=[0, 6, 15, 20],
    labels=["powerplay", "middle", "death"]
)

In [9]:
df_over['overs_completed'] = (120 - df_over['balls_left']) / 6


df_over['wicket_pressure'] = df_over['wickets_fallen'] / (df_over['overs_completed'] + 1)

df_over['wicket_pressure'] = df_over['wicket_pressure'].fillna(0)

In [10]:
df_over["runs_per_wicket"] = df_over["runs_left"] / (df_over["wickets_left"] + 1)
df_over["balls_per_wicket"] = df_over["balls_left"] / (df_over["wickets_left"] + 1)

In [11]:
df_over['momentum']=df_over["current_run_rate"]-df_over['required_run_rate']

In [12]:
balls_bowled = 120 - df_over['balls_left']
    
    # 2. Define the conditions
conditions = [
        (balls_bowled <= 36),                            
        (balls_bowled > 36) & (balls_bowled <= 90),     
        (balls_bowled > 90)                              
    ]
    
values = ['powerplay', 'middle_overs', 'death_overs']
    
df_over['phase'] = np.select(conditions, values, default='middle_overs')


In [13]:
df_over.drop("over",axis=1,inplace=True)

In [14]:
df_over.drop(columns='overs_completed',axis=1,inplace=True)

In [15]:
df_over.drop(columns='pressure',axis=1,inplace=True)

In [16]:
df_over.to_csv("IPL_cleaned_dataset.csv")